# 06 — UCI Dataset Preprocessing

This notebook preprocesses the UCI Heart Disease dataset by aligning its features with the Mendeley dataset, applying the previously fitted preprocessing pipeline, and exporting processed datasets for cross-dataset validation.

In [1]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib


In [2]:

# Load processed Mendeley training features
X_train = pd.read_csv("../data/processed/X_train_scaled.csv")

# Load raw UCI dataset
uci = pd.read_csv("../data/raw/heart_disease_uci.csv")



In [3]:

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)

print(f"Mendeley Training Shape : {X_train.shape}")
print(f"UCI Dataset Shape       : {uci.shape}")

display(uci.head())


DATASET INFORMATION
Mendeley Training Shape : (800, 12)
UCI Dataset Shape       : (920, 16)


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [4]:

uci.info()

display(
    uci.describe(include="all").T
)



<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    str    
 3   dataset   920 non-null    str    
 4   cp        920 non-null    str    
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    str    
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    str    
 13  ca        309 non-null    float64
 14  thal      434 non-null    str    
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(2), str(6)
memory usage: 115.1+ KB


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,920.0,NaN,NaN,NaN,460.5,265.725422,1.0,230.75,460.5,690.25,920.0
age,920.0,NaN,NaN,NaN,53.51087,9.424685,28.0,47.0,54.0,60.0,77.0
sex,920,2,Male,726,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dataset,920,4,Cleveland,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cp,920,4,asymptomatic,496,NaN,NaN,NaN,NaN,NaN,NaN,NaN
trestbps,861.0,NaN,NaN,NaN,132.132404,19.06607,0.0,120.0,130.0,140.0,200.0
chol,890.0,NaN,NaN,NaN,199.130337,110.78081,0.0,175.0,223.0,268.0,603.0
fbs,830,2,False,692,NaN,NaN,NaN,NaN,NaN,NaN,NaN
restecg,918,3,normal,551,NaN,NaN,NaN,NaN,NaN,NaN,NaN
thalch,865.0,NaN,NaN,NaN,137.545665,25.926276,60.0,120.0,140.0,157.0,202.0


In [5]:


print("=" * 60)
print("UNIQUE VALUES")
print("=" * 60)

for column in uci.columns:

    print(f"\n{column}")

    print(
        uci[column].unique()
    )



UNIQUE VALUES

id
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144
 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162
 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180
 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216
 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234
 235 236 237 238 239 240 241 242 

In [6]:

# Invalid blood pressure
uci["trestbps"] = uci["trestbps"].replace(
    0,
    np.nan
)

# Invalid cholesterol
uci["chol"] = uci["chol"].replace(
    0,
    np.nan
)

# Invalid oldpeak
uci.loc[
    uci["oldpeak"] < 0,
    "oldpeak"
] = np.nan


In [7]:

numeric_columns = [
    "age",
    "trestbps",
    "chol",
    "thalch",
    "oldpeak"
]


categorical_columns = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]



In [8]:


print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

display(
    pd.DataFrame({
        "Missing": uci.isnull().sum(),
        "Percentage":
        (
            uci.isnull().mean()*100
        ).round(2)
    })
)


MISSING VALUES


,Missing,Percentage
id,0,0.00
age,0,0.00
sex,0,0.00
dataset,0,0.00
cp,0,0.00
trestbps,60,6.52
chol,202,21.96
fbs,90,9.78
restecg,2,0.22
thalch,55,5.98


In [9]:

print("=" * 60)
print("SKEWNESS")
print("=" * 60)

for column in numeric_columns:

    print(
        f"{column:<15} {uci[column].skew():.3f}"
    )



SKEWNESS
age             -0.196
trestbps        0.630
chol            1.315
thalch          -0.211
oldpeak         1.156


In [10]:
for column in categorical_columns:
    uci[column] = uci[column].fillna(
        uci[column].mode()[0]
    )

In [11]:
uci["age"] = uci["age"].fillna(uci["age"].mean())
uci["trestbps"] = uci["trestbps"].fillna(uci["trestbps"].mean())
uci["thalch"] = uci["thalch"].fillna(uci["thalch"].mean())

uci["chol"] = uci["chol"].fillna(uci["chol"].median())
uci["oldpeak"] = uci["oldpeak"].fillna(uci["oldpeak"].median())

In [12]:

print("=" * 60)
print("VERIFY MISSING VALUES")
print("=" * 60)

display(
    uci.isnull().sum()
)




VERIFY MISSING VALUES


id          0
age         0
sex         0
dataset     0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
num         0
dtype: int64

In [13]:

uci["sex"] = uci["sex"].map({
    "Male":1,
    "Female":0
})

uci["cp"] = uci["cp"].map({
    "typical angina":0,
    "atypical angina":1,
    "non-anginal":2,
    "asymptomatic":3
})

uci["fbs"] = uci["fbs"].map({
    True:1,
    False:0
})

uci["restecg"] = uci["restecg"].map({
    "normal":0,
    "st-t abnormality":1,
    "lv hypertrophy":2
})

uci["exang"] = uci["exang"].map({
    True:1,
    False:0
})

uci["slope"] = uci["slope"].map({
    "upsloping":0,
    "flat":1,
    "downsloping":2
})

uci["thal"] = uci["thal"].map({
    "normal":0,
    "fixed defect":1,
    "reversable defect":2
})


In [14]:

uci["num"] = uci["num"].apply(
    lambda x:0 if x==0 else 1
)



In [15]:

uci.head()


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,1,Cleveland,0,145.0,233.0,1,2,150.0,0,2.3,2,0.0,1,0
1,2,67,1,Cleveland,3,160.0,286.0,0,2,108.0,1,1.5,1,3.0,0,1
2,3,67,1,Cleveland,3,120.0,229.0,0,2,129.0,1,2.6,1,2.0,2,1
3,4,37,1,Cleveland,2,130.0,250.0,0,0,187.0,0,3.5,2,0.0,0,0
4,5,41,0,Cleveland,1,130.0,204.0,0,2,172.0,0,1.4,0,0.0,0,0


In [16]:


for column in categorical_columns:
    uci[column] = uci[column].astype(int)

uci["ca"] = uci["ca"].astype(int)
uci["num"] = uci["num"].astype(int)

uci["age"] = uci["age"].astype(int)
uci["trestbps"] = uci["trestbps"].astype(int)
uci["thalch"] = uci["thalch"].astype(int)


In [17]:

print("=" * 60)
print("DATA TYPES")
print("=" * 60)

display(uci.dtypes)
display(X_train.dtypes)



DATA TYPES


id            int64
age           int64
sex           int64
dataset         str
cp            int64
trestbps      int64
chol        float64
fbs           int64
restecg       int64
thalch        int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
num           int64
dtype: object

age                  float64
gender               float64
chestpain            float64
restingBP            float64
serumcholestrol      float64
fastingbloodsugar    float64
restingrelectro      float64
maxheartrate         float64
exerciseangia        float64
oldpeak              float64
slope                float64
noofmajorvessels     float64
dtype: object

In [18]:


column_mapping = {
    "sex": "gender",
    "cp": "chestpain",
    "trestbps": "restingBP",
    "chol": "serumcholestrol",
    "fbs": "fastingbloodsugar",
    "restecg": "restingrelectro",
    "thalch": "maxheartrate",
    "exang": "exerciseangia",
    "ca": "noofmajorvessels",
    "num": "target"
}


In [19]:

uci.rename(
    columns=column_mapping,
    inplace=True
)



In [20]:


uci_cleveland = uci[
    uci["dataset"] == "Cleveland"
].copy()

# Remaining hospitals -> Training Pool
uci_train = uci[
    uci["dataset"] != "Cleveland"
].copy()


In [21]:

print("Training Pool :", uci_train.shape)
print("Cleveland :", uci_cleveland.shape)



Training Pool : (616, 16)
Cleveland : (304, 16)


In [22]:

columns_to_drop = [
    "id",
    "dataset",
    "thal"
]

uci_train.drop(
    columns=columns_to_drop,
    inplace=True
)

uci_cleveland.drop(
    columns=columns_to_drop,
    inplace=True
)


In [23]:

uci_train = uci_train[
    X_train.columns
]

uci_cleveland = uci_cleveland[
    X_train.columns
]


In [24]:

print("=" * 60)
print("FEATURE ALIGNMENT")
print("=" * 60)

print(uci_train.columns)

print()

print(X_train.columns)


FEATURE ALIGNMENT
Index(['age', 'gender', 'chestpain', 'restingBP', 'serumcholestrol',
       'fastingbloodsugar', 'restingrelectro', 'maxheartrate', 'exerciseangia',
       'oldpeak', 'slope', 'noofmajorvessels'],
      dtype='str')

Index(['age', 'gender', 'chestpain', 'restingBP', 'serumcholestrol',
       'fastingbloodsugar', 'restingrelectro', 'maxheartrate', 'exerciseangia',
       'oldpeak', 'slope', 'noofmajorvessels'],
      dtype='str')


In [25]:

X_uci_train = uci_train.copy()

X_uci_cleveland = uci_cleveland.copy()


In [26]:

y_uci_train = uci.loc[
    uci["dataset"] != "Cleveland",
    "target"
].reset_index(drop=True)

y_uci_cleveland = uci.loc[
    uci["dataset"] == "Cleveland",
    "target"
].reset_index(drop=True)



In [27]:

scaler = joblib.load(
    "../saved_models/scaler.pkl"
)


In [28]:

X_uci_train = pd.DataFrame(
    scaler.transform(X_uci_train),
    columns=X_train.columns
)

X_uci_cleveland = pd.DataFrame(
    scaler.transform(X_uci_cleveland),
    columns=X_train.columns
)



In [29]:

print(X_uci_train.shape)
print(y_uci_train.shape)

print(X_uci_cleveland.shape)
print(y_uci_cleveland.shape)



(616, 12)
(616,)
(304, 12)
(304,)


In [30]:


X_uci_train.to_csv(
    "../data/processed/X_uci_train.csv",
    index=False
)

y_uci_train.to_csv(
    "../data/processed/y_uci_train.csv",
    index=False
)

X_uci_cleveland.to_csv(
    "../data/processed/X_uci_cleveland.csv",
    index=False
)

y_uci_cleveland.to_csv(
    "../data/processed/y_uci_cleveland.csv",
    index=False
)


In [31]:


print("=" * 60)
print("UCI PREPROCESSING COMPLETED")
print("=" * 60)

print("Training Pool Saved")
print("Cleveland Test Set Saved")

print("\nReady for 07_Cross_Dataset_Validation.ipynb")



UCI PREPROCESSING COMPLETED
Training Pool Saved
Cleveland Test Set Saved

Ready for 07_Cross_Dataset_Validation.ipynb
